### Fase 1: Transformando Canciones en "Canastas"
El algoritmo Apriori no lee DataFrames normales; necesita una lista de transacciones. 
En nuestro caso, cada **Canción** es una transacción, y los **Artistas** que participan en ella son los artículos de la canasta.

In [34]:
# Instalar si es necesario: !pip install mlxtend
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import re

# 1. INGESTA DE DATOS
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"
df_spotify = pd.read_csv(url, encoding='latin-1')

# 2. LIMPIEZA: Eliminar nulos de la columna de artistas
df_spotify = df_spotify.dropna(subset=['Artist'])

# 3. TRANSFORMACIÓN: Separar los artistas que colaboran en cada canción
# Asumimos que las colaboraciones están separadas por coma (ej. "Bad Bunny, Feid")
# Convertimos esa columna en una verdadera lista de Python
patron = re.compile(r'[\(\[]\s*(?:feat\.?|ft\.?|with)\s+([^\)\]]+)[\)\]]', re.IGNORECASE)
separadores = re.compile(r'\s*(?:,|&|\band\b|\bf(?:ea)?t\.?(?=\s)|\bwith\b)\s*', re.IGNORECASE)

def extraer_artistas(row):
    artistas = [row['Artist']]
    track = str(row['Track'])
    for m in patron.finditer(track):
        artistas += [a.strip() for a in separadores.split(m.group(1)) if a.strip()]
    return list(dict.fromkeys(artistas))  # remove duplicates, keep order

df_spotify['Artistas'] = df_spotify.apply(extraer_artistas, axis=1)
lista_colaboraciones = df_spotify['Artistas'].tolist()

# Muestra rápida de cómo se ven nuestras "canastas"
print("Muestra de las primeras 5 colaboraciones:")
for i in range(5):
    print(f"Canción {i+1}: {lista_colaboraciones[i]}")

# display(lista_colaboraciones)

# Utiliza TransactionEncoder para convertir 'lista_colaboraciones' en una matriz booleana (One-Hot)
# Guárdala en un DataFrame llamado 'df_transacciones'

te = TransactionEncoder()
matriz_bool = te.fit_transform(lista_colaboraciones)
df_transacciones = pd.DataFrame(matriz_bool, columns=te.columns_)
display(df_transacciones.head())

Muestra de las primeras 5 colaboraciones:
Canción 1: ['Tommy Richman']
Canción 2: ['Kendrick Lamar']
Canción 3: ['Artemas']
Canción 4: ['Miley Cyrus']
Canción 5: ['Eminem']


,"""XY""",$OHO BANI,$uicideboy$,&ME,(G)I-DLE,*NSYNC,.diedlonely,10CM,13 Organisï¿,1da Banton,...,ýýýýýýýýýý,ýýýýýýýýýý ýýýýýýýý,ýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýý,ýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý),ýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý,ýýýýýýýýýýýýýýýýýýýýýýýý
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


### Fase 2: Aplicando Apriori
Ahora que tenemos la matriz con ceros y unos, extraigamos los conjuntos de artistas que colaboran frecuentemente.
Debido a que hay miles de artistas solistas, las colaboraciones son estadísticamente raras. Usaremos un `min_support` muy bajo para capturar las colaboraciones.

In [35]:
# --- RETO PARA EL ALUMNO AQUÍ ---

# 1. Encuentra los Itemsets Frecuentes usando un soporte mínimo del 0.005 (0.5% de las canciones)
itemsets_frecuentes = apriori(df_transacciones, min_support=0.0005, use_colnames=True)

# 2. Genera las Reglas de Asociación filtrando por la métrica 'lift' con un mínimo de 1.2
reglas_musicales = association_rules(itemsets_frecuentes, metric='lift', min_threshold=1.2)
# display(reglas_musicales)

# 3. Muestra las reglas resultantes ordenadas por 'confidence' de mayor a menor
display(reglas_musicales.sort_values(by=['confidence'], ascending=False))

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
5,(Bryant Myers),(Dei V),0.000653,0.001088,0.000653,1.000000,919.000000,1.0,0.000652,inf,0.999564,0.600000,1.000000,0.800000
14,(PARTYNEXTDOOR),(Drake),0.000871,0.018498,0.000653,0.750000,40.544118,1.0,0.000637,3.926007,0.976185,0.034884,0.745288,0.392647
4,(Dei V),(Bryant Myers),0.001088,0.000653,0.000653,0.600000,919.000000,1.0,0.000652,2.498368,1.000000,0.600000,0.599739,0.800000
18,(Sexyy Red),(Drake),0.001306,0.018498,0.000653,0.500000,27.029412,1.0,0.000629,1.963003,0.964262,0.034091,0.490576,0.267647
6,(DJ Khaled),(Drake),0.001523,0.018498,0.000653,0.428571,23.168067,1.0,0.000625,1.717628,0.958297,0.033708,0.417802,0.231933
9,(DJ Snake),(Ozuna),0.001959,0.003917,0.000653,0.333333,85.092593,1.0,0.000645,1.494124,0.990188,0.125000,0.330712,0.250000
2,(Metro Boomin),(21 Savage),0.003482,0.006964,0.000871,0.250000,35.898438,1.0,0.000846,1.324048,0.975541,0.090909,0.244740,0.187500
25,(Lil Durk),(Lil Baby),0.003047,0.007617,0.000653,0.214286,28.132653,1.0,0.000630,1.263033,0.967402,0.065217,0.208255,0.150000
29,(Metro Boomin),(Travis Scott),0.003482,0.009358,0.000653,0.187500,20.036337,1.0,0.000620,1.219252,0.953410,0.053571,0.179825,0.128634
31,(Young Thug),(Travis Scott),0.003700,0.009358,0.000653,0.176471,18.857729,1.0,0.000618,1.202922,0.950488,0.052632,0.168691,0.123119


### 🧠 Fase 3: Interpretación de Negocio (El Entregable)

Como Científico de Datos, no basta con entregar una tabla. Debes explicarle los hallazgos al Director Creativo. Revisa tu DataFrame `reglas_musicales` y responde las siguientes 4 preguntas:

**1. Análisis de Soporte (El Volumen):**
Menciona a los dos artistas que conforman la regla con el mayor **Soporte**. ¿Qué significa este número en términos simples para el negocio?

> **Tu Respuesta:** Metro boomi & 21 Savage y Drake & 21 Savage

**2. Análisis de Confianza (La Dirección):**
Busca una regla que tenga una **Confianza** mayor al 0.5 (50%). Traduce esta regla matemática a una frase que el director pueda entender (Ej. "Si contratamos al artista A, existe un X% de probabilidad de que el artista B también deba estar en el track para que sea el éxito que registramos").

> **Tu Respuesta:** Si contratamos a Bryant Myers, existe un 100% de probabilidad de que Dei V también deba estar en el track para que sea el éxito que registramos

**3. Análisis de Lift (El Impulso):**
Busca la regla con el **Lift** más alto de toda tu tabla. 
¿Por qué esta métrica es más confiable que el Soporte para decidir si estos dos artistas *realmente* tienen química musical o si solo fue una coincidencia estadística?

> **Tu Respuesta:** Porque el lift brinda una métrica donde sí se toma en cuenta la influencia directa de una variable sobre la otra, mientras que el support muestra más la pura frecuencia/popularidad de variables sin ver realmente relaciones, simplemente brinda la cantidad de veces que las variables estuvieron juntas, no que tanto actúan una sobre la otra.

**4. Toma de Decisión:**
Basado puramente en las reglas matemáticas que descubriste, si la disquera solo tiene presupuesto para juntar a dos artistas en el estudio la próxima semana, ¿qué pareja elegirías y por qué? (Justifica usando las 3 métricas de oro).

> **Tu Respuesta:** Elegiría a Metro boomin y 21 Savage, se trata de una regla que se encuentra bien posicionada para las tres métricas clave a comparación de otras combinaciones representadas, además que los valores de las mismas se encuentran en rangos que hacen sentido a comparación de otros, por jemplo Bryant Myers y Dei V que tuvieron un lift de 919